In [ ]:
from __future__ import annotations

import io
import json
import math
import re
import statistics
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Callable, Iterable
from xml.etree import ElementTree as ET

import fitz  # PyMuPDF
import pandas as pd

try:
    import pdfplumber
except Exception:
    pdfplumber = None

try:
    from PIL import Image
except Exception:
    Image = None
    print("PIL not found. Image processing features will be unavailable.")

try:
    import pytesseract
except Exception:
    pytesseract = None

## **Configuration**

In [ ]:
@dataclass
class IngestionConfig:
    ROOT = Path.cwd().parent
    input_dir: str = str(ROOT / "data" / "test")
    output_dir: str = str(ROOT / "data" / "processed_reports")
    rendered_dpi: int = 220
    min_text_chars: int = 80
    min_alpha_ratio: float = 0.55
    min_words: int = 15
    image_heavy_threshold: float = 0.45
    max_chunk_chars: int = 1500
    chunk_overlap: int = 150
    ocr_enabled: bool = False
    save_rendered_pages: bool = False
    supported_pdf_exts: tuple[str, ...] = (".pdf",)
    supported_sheet_exts: tuple[str, ...] = (".xlsx", ".xlsm", ".xls")
    supported_csv_exts: tuple[str, ...] = (".csv",)
    supported_doc_exts: tuple[str, ...] = (".docx",)

CFG = IngestionConfig()
BASE_INPUT_DIR = Path(CFG.input_dir)
BASE_OUTPUT_DIR = Path(CFG.output_dir)

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(BASE_OUTPUT_DIR / "page_images").mkdir(parents=True, exist_ok=True)
(BASE_OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
(BASE_OUTPUT_DIR / "chunks").mkdir(parents=True, exist_ok=True)

#print(CFG)

## **Helper functions**

### **General utilities**

In [ ]:
def discover_files(root: str | Path) -> list[Path]:
    root = Path(root)
    exts = set(CFG.supported_pdf_exts + CFG.supported_sheet_exts + CFG.supported_csv_exts + CFG.supported_doc_exts)
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts])


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def alpha_ratio(text: str) -> float:
    if not text:
        return 0.0
    alpha = sum(ch.isalpha() for ch in text)
    return alpha / max(len(text), 1)


def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))


def chunk_text(text: str, max_chars: int = 1500, overlap: int = 150) -> list[str]:
    text = clean_text(text)
    if not text:
        return []

    chunks: list[str] = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]

        # try to break on a paragraph or sentence boundary
        if end < n:
            last_break = max(chunk.rfind("\n\n"), chunk.rfind(". "), chunk.rfind("\n"))
            if last_break > max_chars * 0.6:
                end = start + last_break + 1
                chunk = text[start:end]

        chunks.append(chunk.strip())

        if end >= n:
            break
        start = max(end - overlap, start + 1)

    return [c for c in chunks if c]


def normalize_column_name(col: Any) -> str:
    col = "" if col is None else str(col)
    col = clean_text(col)
    col = re.sub(r"[^0-9A-Za-z_\-%/(). ]+", "", col)
    col = re.sub(r"\s+", "_", col.strip().lower())
    return col or "unnamed"


def coerce_numeric(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (int, float)) and not pd.isna(value):
        return value

    s = str(value).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None

    s = s.replace(",", "")
    s = s.replace("−", "-")

    pct = s.endswith("%")
    if pct:
        s = s[:-1].strip()

    negative_paren = s.startswith("(") and s.endswith(")")
    if negative_paren:
        s = s[1:-1].strip()

    try:
        val = float(s)
        if negative_paren:
            val = -val
        return val / 100.0 if pct else val
    except Exception:
        return value


def detect_units_from_text(text: str) -> str | None:
    txt = (text or "").lower()
    patterns = [
        ("billions", r"\bbillions?\b|\b\$b\b"),
        ("millions", r"\bmillions?\b|\b\$m\b"),
        ("percent", r"\bpercent\b|\b%\b"),
        ("usd", r"\busd\b|\$"),
        ("eur", r"\beur\b|€"),
    ]
    for label, pattern in patterns:
        if re.search(pattern, txt):
            return label
    return None


def first_match(text: str, patterns: Iterable[str]) -> str | None:
    for pattern in patterns:
        match = re.search(pattern, text or "", flags=re.IGNORECASE)
        if match:
            return match.group(1) if match.groups() else match.group(0)
    return None


def infer_document_metadata(path: Path, text: str = "") -> dict[str, Any]:
    context = clean_text(f"{path.stem} {text}")
    lower_context = context.lower()

    known_companies = {
        "microsoft": "Microsoft",
        "msft": "Microsoft",
        "apple": "Apple",
        "aapl": "Apple",
        "alphabet": "Alphabet",
        "google": "Alphabet",
        "googl": "Alphabet",
        "amazon": "Amazon",
        "amzn": "Amazon",
        "nvidia": "NVIDIA",
        "nvda": "NVIDIA",
    }
    company = next((label for key, label in known_companies.items() if key in lower_context), None)

    fiscal_year = first_match(context, [r"\bFY\s?(20\d{2}|\d{2})\b", r"\bfiscal year\s+(20\d{2})\b"])
    if fiscal_year and len(fiscal_year) == 2:
        fiscal_year = f"20{fiscal_year}"

    fiscal_quarter = first_match(context, [r"\b(Q[1-4])\b", r"\bquarter\s+([1-4])\b"])
    if fiscal_quarter and fiscal_quarter.isdigit():
        fiscal_quarter = f"Q{fiscal_quarter}"

    period_parts = [part for part in [fiscal_year, fiscal_quarter] if part]
    period = " ".join(period_parts) or first_match(context, [r"\b(20\d{2})\b"])

    currency = first_match(context, [r"\b(USD|EUR|GBP)\b", r"(\$)", r"(€)"])
    if currency == "$":
        currency = "USD"
    elif currency == "€":
        currency = "EUR"

    scale = detect_units_from_text(context)

    return {
        "document_title": path.stem.replace("_", " ").replace("-", " "),
        "company": company,
        "fiscal_year": fiscal_year,
        "fiscal_quarter": fiscal_quarter,
        "period": period,
        "currency": currency,
        "scale": scale,
    }


def infer_section_heading(text: str) -> str | None:
    for line in clean_text(text).splitlines()[:8]:
        line = line.strip()
        if 4 <= len(line) <= 100 and word_count(line) <= 12:
            if line.isupper() or re.match(r"^[A-Z][A-Za-z0-9 ,:&/().-]+$", line):
                return line
    return None


def make_retrieval_text(record: dict[str, Any], fields: Iterable[str]) -> str:
    parts = []
    for field in fields:
        value = record.get(field)
        if value is not None and value != "":
            parts.append(f"{field}: {value}")
    return " | ".join(parts)


def write_dataframe(df: pd.DataFrame, parquet_path: Path) -> Path:
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as e:
        csv_path = parquet_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        print(f"Could not write parquet ({type(e).__name__}: {e}); wrote CSV instead: {csv_path}")
        return csv_path




### **PDF page analysis**

In [ ]:
def extract_native_page_text(page: fitz.Page) -> str:
    return clean_text(page.get_text("text"))


def estimate_image_coverage(page: fitz.Page) -> float:
    # Approximate how much of the page area is occupied by images.
    page_area = page.rect.width * page.rect.height
    if page_area <= 0:
        return 0.0

    coverage = 0.0
    try:
        image_infos = page.get_image_info()
    except Exception:
        image_infos = []

    for info in image_infos:
        bbox = info.get("bbox")
        if not bbox:
            continue
        x0, y0, x1, y1 = bbox
        coverage += max(0, x1 - x0) * max(0, y1 - y0)

    return min(coverage / page_area, 1.0)


def classify_pdf_page(page: fitz.Page) -> dict[str, Any]:
    text = extract_native_page_text(page)
    n_chars = len(text)
    n_words = word_count(text)
    a_ratio = alpha_ratio(text)
    image_ratio = estimate_image_coverage(page)

    weak_text = (
        n_chars < CFG.min_text_chars or
        n_words < CFG.min_words or
        a_ratio < CFG.min_alpha_ratio
    )

    if weak_text and image_ratio >= CFG.image_heavy_threshold:
        page_type = "image_native"
    elif weak_text:
        page_type = "weak_text"
    elif image_ratio >= CFG.image_heavy_threshold:
        page_type = "mixed_or_image_heavy"
    else:
        page_type = "text_native"

    return {
        "page_type": page_type,
        "native_text": text,
        "n_chars": n_chars,
        "n_words": n_words,
        "alpha_ratio": round(a_ratio, 4),
        "image_ratio": round(image_ratio, 4),
        "weak_text": weak_text,
    }


def render_page(page: fitz.Page, dpi: int = 220) -> Image.Image | None:
    if Image is None:
        raise RuntimeError("Pillow is not installed.")
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def maybe_save_page_image(image: Image.Image, pdf_path: Path, page_num: int) -> Path:
    out_path = BASE_OUTPUT_DIR / "page_images" / f"{pdf_path.stem}_page_{page_num+1}.png"
    image.save(out_path)
    return out_path


## **Extraction backends**

### **OCR / vision fallback**

In [ ]:
def ocr_image(image: Image.Image) -> str: # type: ignore
    if not CFG.ocr_enabled:
        return ""
    if pytesseract is None:
        return ""
    return clean_text(pytesseract.image_to_string(image))


def default_vision_extractor(image: Image.Image, *, source_hint: dict[str, Any] | None = None) -> dict[str, Any]: # type: ignore
    # Placeholder integration point.
    return {
        "page_text": "",
        "tables": [],
        "confidence": None,
        "source_hint": source_hint or {},
        "note": "No vision extractor configured."
    }


### **Table extraction for text-native PDFs**

In [ ]:
def extract_pdf_tables_text_native(pdf_path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    if pdfplumber is None:
        return records

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            try:
                tables = page.extract_tables()
            except Exception:
                tables = []

            for table_idx, table in enumerate(tables):
                if not table:
                    continue
                df = pd.DataFrame(table)
                if df.empty:
                    continue

                # Treat first row as header when it looks plausible
                if len(df) > 1:
                    header = [normalize_column_name(c) for c in df.iloc[0].tolist()]
                    if len(set(header)) > 1:
                        df = df[1:].copy()
                        df.columns = header
                    else:
                        df.columns = [f"col_{i}" for i in range(df.shape[1])]
                else:
                    df.columns = [f"col_{i}" for i in range(df.shape[1])]

                df = df.dropna(how="all")
                if df.empty:
                    continue

                for row_num, row in df.reset_index(drop=True).iterrows():
                    row_dict = {str(k): (None if pd.isna(v) else str(v).strip()) for k, v in row.items()}
                    row_context = " ".join(str(v) for v in row_dict.values() if v is not None)
                    records.append({
                        "source_file": str(pdf_path),
                        "source_type": "pdf_table_native",
                        **infer_document_metadata(pdf_path, row_context),
                        "page_num": page_idx + 1,
                        "table_idx": table_idx,
                        "row_idx": row_num,
                        "table_name": f"{pdf_path.stem}_p{page_idx+1}_t{table_idx}",
                        "table_context": row_context,
                        "row_data": row_dict
                    })

    return records


### **Spreadsheet / CSV ingestion**

In [ ]:
def read_tabular_file(path: Path) -> dict[str, pd.DataFrame]:
    ext = path.suffix.lower()

    if ext in CFG.supported_csv_exts:
        return {"csv": pd.read_csv(path)}

    if ext in CFG.supported_sheet_exts:
        xl = pd.ExcelFile(path)
        return {sheet: xl.parse(sheet) for sheet in xl.sheet_names}

    raise ValueError(f"Unsupported file type: {path}")


def dataframe_to_records(
    df: pd.DataFrame,
    *,
    source_file: Path,
    sheet_name: str,
    source_type: str,
    extra_meta: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    df = df.copy()
    df.columns = [normalize_column_name(c) for c in df.columns]
    df = df.dropna(how="all")
    if df.empty:
        return []

    records: list[dict[str, Any]] = []
    for row_idx, row in df.reset_index(drop=True).iterrows():
        row_dict = {col: (None if pd.isna(val) else val) for col, val in row.items()}
        row_context = " ".join(str(v) for v in row_dict.values() if v is not None)
        records.append({
            "source_file": str(source_file),
            "source_type": source_type,
            "sheet_name": sheet_name,
            "table_name": f"{source_file.stem}_{sheet_name}",
            "row_idx": row_idx,
            "table_context": row_context,
            **(extra_meta or {}),
            "row_data": row_dict
        })
    return records


## **Normalization**



### **Table normalization**

In [ ]:
def wide_row_records_to_long(records: list[dict[str, Any]]) -> pd.DataFrame:
    long_rows: list[dict[str, Any]] = []

    for rec in records:
        row_data = rec.get("row_data", {}) or {}
        if not row_data:
            continue

        # Heuristic: use the first non-numeric-looking column as row label
        label_col = None
        for k, v in row_data.items():
            if isinstance(v, str) and coerce_numeric(v) == v:
                label_col = k
                break

        row_label = row_data.get(label_col) if label_col else None
        inferred_units = detect_units_from_text(" ".join(map(str, row_data.values())))

        for col, raw_val in row_data.items():
            val = coerce_numeric(raw_val)
            if isinstance(val, (int, float)) and not pd.isna(val):
                long_rows.append({
                    "source_file": rec.get("source_file"),
                    "source_type": rec.get("source_type"),
                    "document_title": rec.get("document_title"),
                    "company": rec.get("company"),
                    "fiscal_year": rec.get("fiscal_year"),
                    "fiscal_quarter": rec.get("fiscal_quarter"),
                    "period": rec.get("period"),
                    "currency": rec.get("currency"),
                    "scale": rec.get("scale"),
                    "sheet_name": rec.get("sheet_name"),
                    "page_num": rec.get("page_num"),
                    "table_idx": rec.get("table_idx"),
                    "table_name": rec.get("table_name"),
                    "table_context": rec.get("table_context"),
                    "row_idx": rec.get("row_idx"),
                    "row_label": row_label,
                    "metric": col,
                    "value": val,
                    "units": inferred_units or rec.get("scale"),
                    "raw_value": raw_val,
                })

    long_df = pd.DataFrame(long_rows)
    if not long_df.empty:
        long_df["retrieval_text"] = long_df.apply(
            lambda row: make_retrieval_text(row.to_dict(), [
                "document_title", "company", "period", "table_name", "row_label",
                "metric", "value", "units", "currency", "table_context"
            ]),
            axis=1,
        )
    return long_df


### **Unstructured chunk packaging**

In [ ]:
def build_text_chunk_records(
    *,
    text: str,
    source_file: Path,
    page_num: int | None = None,
    source_type: str = "text",
    extra_meta: dict[str, Any] | None = None,
) -> list[dict[str, Any]]:
    chunks = chunk_text(text, max_chars=CFG.max_chunk_chars, overlap=CFG.chunk_overlap)
    out = []
    for idx, chunk in enumerate(chunks):
        semantic_meta = infer_document_metadata(source_file, chunk)
        rec = {
            "source_file": str(source_file),
            "source_type": source_type,
            "content_type": "unstructured_text",
            **semantic_meta,
            "page_num": page_num,
            "chunk_idx": idx,
            "section_heading": infer_section_heading(chunk),
            "n_chars": len(chunk),
            "n_words": word_count(chunk),
            "alpha_ratio": round(alpha_ratio(chunk), 4),
            "extraction_status": "text_extracted",
            "text": chunk,
        }
        if extra_meta:
            rec.update(extra_meta)
        rec["retrieval_text"] = make_retrieval_text(rec, [
            "document_title", "company", "period", "section_heading", "source_type", "text"
        ])
        out.append(rec)
    return out


def build_image_page_record(
    *,
    source_file: Path,
    page_num: int,
    rendered_image: str | None,
    page_type: str,
    extra_meta: dict[str, Any] | None = None,
) -> dict[str, Any]:
    rec = {
        "source_file": str(source_file),
        "source_type": "pdf_rendered_image",
        "content_type": "image_page",
        **infer_document_metadata(source_file),
        "page_num": page_num,
        "chunk_idx": 0,
        "page_type": page_type,
        "rendered_image": rendered_image,
        "text": "",
        "extraction_status": "image_only_no_text_extracted",
    }
    if extra_meta:
        rec.update(extra_meta)
    rec["retrieval_text"] = make_retrieval_text(rec, [
        "document_title", "company", "period", "source_type", "page_type", "rendered_image"
    ])
    return rec


## **Ingestion orchestrator**




### **PDF ingestion orchestration**

In [ ]:
def ingest_pdf(
    pdf_path: Path,
    *,
    vision_extractor: Callable[[Image.Image], dict[str, Any]] | None = None,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], pd.DataFrame]:
    vision_extractor = vision_extractor or default_vision_extractor

    text_chunks: list[dict[str, Any]] = []
    page_records: list[dict[str, Any]] = []
    table_records: list[dict[str, Any]] = []

    doc = fitz.open(pdf_path)

    for page_idx in range(len(doc)):
        page = doc.load_page(page_idx)
        analysis = classify_pdf_page(page)
        page_meta = infer_document_metadata(pdf_path, analysis["native_text"])

        page_rec = {
            "source_file": str(pdf_path),
            "source_type": "pdf_page",
            "record_type": "pdf_page_manifest",
            "page_num": page_idx + 1,
            **page_meta,
            "section_heading": infer_section_heading(analysis["native_text"]),
            **analysis
        }

        page_text = analysis["native_text"]
        page_source_type = "pdf_native_text"

        # Image/OCR/vision fallback is intentionally disabled for now.
        # The helper functions remain available for later image-only PDF support.
        page_rec["extraction_status"] = "text_extracted" if page_text else "no_native_text_extracted"

        # Package chunkable page text
        chunk_meta = {
            **page_meta,
            "page_type": analysis["page_type"],
            "image_ratio": analysis["image_ratio"],
            "alpha_ratio": analysis["alpha_ratio"],
            "extraction_status": page_rec["extraction_status"],
            "record_type": "text_chunk",
        }
        if page_text:
            text_chunks.extend(build_text_chunk_records(
                text=page_text,
                source_file=pdf_path,
                page_num=page_idx + 1,
                source_type=page_source_type,
                extra_meta=chunk_meta,
            ))

        page_records.append(page_rec)

    doc.close()

    # Native text-table extraction
    native_table_records = extract_pdf_tables_text_native(pdf_path)
    table_records.extend(native_table_records)

    long_df = wide_row_records_to_long(table_records)

    return text_chunks, page_records, long_df


### **DOCX ingestion orchestration**

In [ ]:
def read_docx_text(path: Path) -> str:
    text_parts: list[str] = []
    xml_parts = ["word/document.xml"]

    with zipfile.ZipFile(path) as docx_zip:
        xml_parts.extend(
            name for name in docx_zip.namelist()
            if re.match(r"word/(header|footer)\d+\.xml$", name)
        )

        for xml_name in xml_parts:
            if xml_name not in docx_zip.namelist():
                continue

            root = ET.fromstring(docx_zip.read(xml_name))
            ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}

            for paragraph in root.findall(".//w:p", ns):
                paragraph_text = "".join(
                    node.text or ""
                    for node in paragraph.findall(".//w:t", ns)
                )
                paragraph_text = clean_text(paragraph_text)
                if paragraph_text:
                    text_parts.append(paragraph_text)

    return clean_text("\n".join(text_parts))


def ingest_docx(path: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]], pd.DataFrame]:
    text = read_docx_text(path)
    doc_meta = infer_document_metadata(path, text)

    manifest = [{
        "source_file": str(path),
        "source_type": "docx_text",
        "record_type": "docx_manifest",
        **doc_meta,
        "section_heading": infer_section_heading(text),
        "n_chars": len(text),
        "n_words": word_count(text),
        "alpha_ratio": round(alpha_ratio(text), 4),
        "extraction_status": "text_extracted" if text else "no_text_extracted",
    }]

    text_chunks = build_text_chunk_records(
        text=text,
        source_file=path,
        source_type="docx_text",
        extra_meta={**doc_meta, "record_type": "text_chunk"},
    )

    return text_chunks, manifest, pd.DataFrame()


### **Workbook / CSV ingestion orchestration**

In [ ]:
def ingest_tabular_file(path: Path) -> tuple[list[dict[str, Any]], list[dict[str, Any]], pd.DataFrame]:
    text_chunks: list[dict[str, Any]] = []
    sheet_records: list[dict[str, Any]] = []
    table_records: list[dict[str, Any]] = []

    sheets = read_tabular_file(path)
    source_type = "excel_sheet" if path.suffix.lower() in CFG.supported_sheet_exts else "csv"

    for sheet_name, df in sheets.items():
        df_clean = df.dropna(how="all").copy()
        df_clean.columns = [normalize_column_name(c) for c in df_clean.columns]
        preview_text = df_clean.head(20).to_csv(index=False) if not df_clean.empty else ""
        sheet_meta = infer_document_metadata(path, f"{sheet_name} {' '.join(df_clean.columns)} {preview_text}")

        sheet_records.append({
            "source_file": str(path),
            "source_type": source_type,
            "record_type": "sheet_manifest",
            **sheet_meta,
            "sheet_name": sheet_name,
            "n_rows": int(df_clean.shape[0]),
            "n_cols": int(df_clean.shape[1]),
            "columns": list(df_clean.columns),
        })

        table_records.extend(dataframe_to_records(
            df_clean,
            source_file=path,
            sheet_name=sheet_name,
            source_type=source_type,
            extra_meta=sheet_meta,
        ))

    long_df = wide_row_records_to_long(table_records)
    return text_chunks, sheet_records, long_df



## **End-to-end batch ingestion**

Outputs three artifacts:

- `page_or_sheet_manifest.parquet` or `.csv`
- `text_chunks.parquet` or `.csv`
- `structured_long.parquet` or `.csv`

Active ingestion currently focuses on text-native PDFs, DOCX files, spreadsheets, and CSV files. Image/OCR/vision helper functions are kept in the notebook for later, but the PDF path does not call them by default.

`text_chunks` is for unstructured text from DOCX and PDFs. CSV/Excel files are kept in `structured_long` instead of being duplicated as text chunks.

The chunks and structured rows include retrieval metadata such as document title, company, fiscal period, currency, scale, source location, and a `retrieval_text` field for later indexing.



In [ ]:
# Main ingestion function
def ingest_path(
    root: str | Path,
    *,
    vision_extractor: Callable[[Image.Image], dict[str, Any]] | None = None,
) -> dict[str, pd.DataFrame]:
    files = discover_files(root)

    all_chunks: list[dict[str, Any]] = []
    all_manifest: list[dict[str, Any]] = []
    all_structured: list[pd.DataFrame] = []

    for path in files:
        ext = path.suffix.lower()
        print(f"Ingesting: {path}")

        if ext in CFG.supported_pdf_exts:
            chunks, manifest, long_df = ingest_pdf(path, vision_extractor=vision_extractor)
        elif ext in CFG.supported_sheet_exts or ext in CFG.supported_csv_exts:
            chunks, manifest, long_df = ingest_tabular_file(path)
        elif ext in CFG.supported_doc_exts:
            chunks, manifest, long_df = ingest_docx(path)
        else:
            print(f"Skipping unsupported file: {path}")
            continue

        all_chunks.extend(chunks)
        all_manifest.extend(manifest)
        if not long_df.empty:
            all_structured.append(long_df)

    chunks_df = pd.DataFrame(all_chunks)
    manifest_df = pd.DataFrame(all_manifest)
    structured_df = pd.concat(all_structured, ignore_index=True) if all_structured else pd.DataFrame()

    # Persist
    chunks_path = BASE_OUTPUT_DIR / "chunks" / "text_chunks.parquet"
    manifest_path = BASE_OUTPUT_DIR / "page_or_sheet_manifest.parquet"
    structured_path = BASE_OUTPUT_DIR / "tables" / "structured_long.parquet"

    written_paths = {
        "chunks": write_dataframe(chunks_df, chunks_path),
        "manifest": write_dataframe(manifest_df, manifest_path),
    }
    if not structured_df.empty:
        written_paths["structured"] = write_dataframe(structured_df, structured_path)
    else:
        structured_df = pd.DataFrame(columns=[
            "source_file", "source_type", "document_title", "company", "fiscal_year",
            "fiscal_quarter", "period", "currency", "scale", "sheet_name", "page_num",
            "table_idx", "table_name", "table_context", "row_idx", "row_label",
            "metric", "value", "units", "raw_value", "retrieval_text"
        ])
        written_paths["structured"] = write_dataframe(structured_df, structured_path)

    return {
        "manifest": manifest_df,
        "chunks": chunks_df,
        "structured": structured_df,
        "paths": written_paths,
    }


In [ ]:
ingest_path(BASE_INPUT_DIR)